# CNN_5 Exp 2b: Exp 2 + วิธีเทรนแบบ Exp 1 (7 ช่อง, ไม่ทำ Gaussian)

ต่อยอดจาก Exp 2 (AveragePooling (1,25) ก่อน Flatten) เปลี่ยนอย่างเดียว+ ใช้วิธีเทรนแบบ Exp 1c คือ ReduceLROnPlateau และ max epoch 60 (เปลี่ยน 2 ส่วนพร้อมกัน เหมือน Exp 1c)

## 1. Import

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.signal import resample_poly

import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_fscore_support,
)

## 2. ตั้งค่า

In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/SleepStageClassificationProject")
DATA_DIR = PROJECT_DIR / "dataset" / "preprocess_microvolt"
RESULT_DIR = PROJECT_DIR / "results" / "CNN_5_7ch_exp2b"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

FS_RAW = 256
FS = 100
EPOCH_LEN_RAW = 30 * FS_RAW
EPOCH_LEN = 30 * FS
SEQ_LEN = 5
SEQ_SAMPLES = SEQ_LEN * EPOCH_LEN

ALL_CHANNELS = ["F4-M1", "C4-M1", "O2-M1", "C3-M2", "E1-M2", "E2-M2", "Chin"]
SELECTED_CHANNELS = ["F4-M1", "C4-M1", "O2-M1", "C3-M2", "E1-M2", "E2-M2", "Chin"]
CHANNEL_IDX = [ALL_CHANNELS.index(c) for c in SELECTED_CHANNELS]
N_ALL_CHANNELS = len(ALL_CHANNELS)
N_CHANNELS = len(SELECTED_CHANNELS)

STAGES = ["W", "N1", "N2", "N3", "R"]
LABELS = [0, 1, 2, 3, 4]
SPLITS = ["train", "val", "test"]
EXPECTED_SUBJECTS = {"train": 79, "val": 21, "test": 23}

CNN_FILTERS = [8, 16, 32]
KERNEL_SIZE = 100
DENSE_UNITS = 50
EXTRA_POOL = 25
DROPOUT = 0.5

BATCH_SIZE = 80
MAX_TRAIN_EPOCHS = 60
EVALS_PER_EPOCH = 5
LEARNING_RATE = 0.001
LR_FACTOR = 0.3
LR_PATIENCE = 5
MIN_LR = 1e-6
PATIENCE = 10

## 3. โหลดข้อมูล (เฉพาะ subject ที่มีครบ 5 ระยะ ระยะละ 5 epoch ติดกัน)

In [ ]:
def load_split(split):
    X_list = []
    rows = []

    for path in sorted((DATA_DIR / split).glob("*.npz")):
        subject = path.stem
        subject_X = []
        subject_rows = []

        with np.load(path, allow_pickle=True) as data:
            stages = data["stage"]
            n_valid = data["n_valid"]
            y_all = data["y"]
            epochs_all = data["epoch_indices"]
            x_all = data["X"]

            for g in range(len(stages)):
                stage = str(stages[g])

                if stage not in STAGES:
                    continue
                if int(n_valid[g]) != SEQ_LEN:
                    continue

                y_group = np.asarray(y_all[g]).astype(str).ravel()
                epochs = np.asarray(epochs_all[g]).astype(int).ravel()

                if len(y_group) != SEQ_LEN or not np.all(y_group == stage):
                    continue
                if len(epochs) != SEQ_LEN or not np.all(np.diff(epochs) == 1):
                    continue

                x = np.asarray(x_all[g], dtype=np.float32)

                if x.shape != (SEQ_LEN, EPOCH_LEN_RAW, N_ALL_CHANNELS):
                    continue
                if not np.isfinite(x).all():
                    continue

                x = x[:, :, CHANNEL_IDX]
                x = x.reshape(SEQ_LEN * EPOCH_LEN_RAW, N_CHANNELS)
                x = resample_poly(x, FS, FS_RAW, axis=0)

                subject_X.append(x.astype(np.float32))
                subject_rows.append({
                    "subject": subject,
                    "stage": stage,
                    "label": STAGES.index(stage),
                    "epoch_start": int(epochs[0]),
                    "epoch_end": int(epochs[-1]),
                })

        found_stages = set(row["stage"] for row in subject_rows)
        if found_stages == set(STAGES):
            X_list.extend(subject_X)
            rows.extend(subject_rows)

    return np.stack(X_list), pd.DataFrame(rows)

DATA = {}
for split in SPLITS:
    X, meta = load_split(split)
    DATA[split] = {"X": X, "y": meta["label"].to_numpy().astype(np.int32), "meta": meta}
    print(split, "| subjects:", meta["subject"].nunique(), "| sequences:", len(meta), "| X shape:", X.shape)

actual_subjects = {split: DATA[split]["meta"]["subject"].nunique() for split in SPLITS}
assert actual_subjects == EXPECTED_SUBJECTS, actual_subjects

for split in SPLITS:
    meta = DATA[split]["meta"]
    assert meta.groupby(["subject", "stage"]).size().eq(1).all(), split
    assert meta.groupby("subject").size().eq(len(STAGES)).all(), split

for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    assert not set(DATA[a]["meta"]["subject"]) & set(DATA[b]["meta"]["subject"])

X_train, y_train = DATA["train"]["X"], DATA["train"]["y"]
X_val, y_val = DATA["val"]["X"], DATA["val"]["y"]
X_test, y_test = DATA["test"]["X"], DATA["test"]["y"]

display(pd.DataFrame({
    "channel": SELECTED_CHANNELS,
    "min": X_train.min(axis=(0, 1)),
    "max": X_train.max(axis=(0, 1)),
    "mean": X_train.mean(axis=(0, 1)),
    "sd": X_train.std(axis=(0, 1)),
}))

## 4. สร้างโมเดล CNN_5

In [ ]:
def build_cnn5():
    inputs = layers.Input(shape=(SEQ_SAMPLES, N_CHANNELS))
    x = layers.Permute((2, 1))(inputs)
    x = layers.Reshape((N_CHANNELS, SEQ_SAMPLES, 1))(x)

    for n_filters in CNN_FILTERS:
        x = layers.Conv2D(n_filters, (1, KERNEL_SIZE), strides=1, padding="same")(x)
        x = layers.ReLU()(x)
        x = layers.BatchNormalization()(x)
        x = layers.AveragePooling2D(pool_size=(1, 2), strides=(1, 2))(x)

    x = layers.AveragePooling2D(pool_size=(1, EXTRA_POOL), strides=(1, EXTRA_POOL))(x)
    x = layers.Flatten()(x)
    x = layers.Dense(DENSE_UNITS)(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(DROPOUT)(x)
    outputs = layers.Dense(len(STAGES), activation="softmax")(x)

    return models.Model(inputs, outputs, name="CNN_5")


model = build_cnn5()
model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

## 5. ฝึกโมเดล (เลือก checkpoint จาก validation loss, ลด LR เมื่อ validation loss ไม่ดีขึ้น)

In [ ]:
n_train = len(y_train)
batches_per_epoch = int(np.ceil(n_train / BATCH_SIZE))
eval_every = max(1, round(batches_per_epoch / EVALS_PER_EPOCH))

evals_per_epoch = batches_per_epoch / eval_every

print("batches per training epoch:", batches_per_epoch)
print("validate every", eval_every, "batch(es)")
print("validations per training epoch:", evals_per_epoch)
print("patience:", PATIENCE, "validations =", PATIENCE / evals_per_epoch, "training epochs")

history = []
best_val_loss = np.inf
best_weights = model.get_weights()
wait = 0
lr_wait = 0
lr = LEARNING_RATE
step = 0
stop = False

for epoch in range(MAX_TRAIN_EPOCHS):
    order = np.random.permutation(n_train)

    for b in range(batches_per_epoch):
        idx = order[b * BATCH_SIZE:(b + 1) * BATCH_SIZE]
        result = model.train_on_batch(X_train[idx], y_train[idx], return_dict=True)
        step += 1

        if step % eval_every == 0:
            val_loss, val_acc = model.evaluate(X_val, y_val, batch_size=BATCH_SIZE, verbose=0)
            history.append({
                "step": step,
                "train_epoch": step / batches_per_epoch,
                "lr": lr,
                "train_last_batch_loss": result["loss"],
                "train_last_batch_accuracy": result["accuracy"],
                "val_loss": val_loss,
                "val_accuracy": val_acc,
            })

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_weights = model.get_weights()
                wait = 0
                lr_wait = 0
            else:
                wait += 1
                lr_wait += 1

            if lr_wait >= LR_PATIENCE:
                lr = max(lr * LR_FACTOR, MIN_LR)
                model.optimizer.learning_rate.assign(lr)
                lr_wait = 0

            if wait >= PATIENCE:
                stop = True
                break

    print(f"epoch {epoch + 1}: val_loss={history[-1]['val_loss']:.4f}, val_acc={history[-1]['val_accuracy']:.4f}, best_val_loss={best_val_loss:.4f}, lr={lr:.6f}")

    if stop:
        print("early stopping")
        break

model.set_weights(best_weights)
model.save(RESULT_DIR / "cnn5_best_model.keras")

history_df = pd.DataFrame(history)
history_df.to_csv(RESULT_DIR / "training_history.csv", index=False)

best_step = history_df.loc[history_df["val_loss"].idxmin(), "step"]
print("best checkpoint at step", int(best_step), "| val_loss", round(best_val_loss, 4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, name in zip(axes, ["loss", "accuracy"]):
    ax.plot(history_df["step"], history_df[f"train_last_batch_{name}"], label="train (last batch)")
    ax.plot(history_df["step"], history_df[f"val_{name}"], label="validation")
    ax.axvline(best_step, color="gray", linestyle="--", label="best checkpoint")
    ax.set_xlabel("Training update step")
    ax.set_ylabel(name)
    ax.legend()
    ax.grid(True)
plt.tight_layout()
plt.savefig(RESULT_DIR / "training_curves.png", dpi=150)
plt.show()

## 6. ฟังก์ชันคำนวณ metric

In [ ]:
def calc_metrics(y_true, y_pred):
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=LABELS, zero_division=0
    )
    overall = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": precision.mean(),
        "macro_recall": recall.mean(),
        "macro_f1": f1.mean(),
        "cohen_kappa": cohen_kappa_score(y_true, y_pred, labels=LABELS),
    }
    per_class = pd.DataFrame({
        "stage": STAGES,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "support": support,
    })
    return overall, per_class


def stage_confusion(cm):
    rows = []
    total = cm.sum()
    for i, stage in enumerate(STAGES):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = total - tp - fp - fn
        rows.append({
            "stage": stage,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "TN": tn,
            "sensitivity": tp / (tp + fn) if tp + fn > 0 else 0,
            "specificity": tn / (tn + fp) if tn + fp > 0 else 0,
        })
    return pd.DataFrame(rows)


def class_analysis(cm):
    rows = []
    for i, stage in enumerate(STAGES):
        errors = cm[i].copy()
        errors[i] = 0
        j = errors.argmax()
        rows.append({
            "stage": stage,
            "correct": int(cm[i, i]),
            "total": int(cm[i].sum()),
            "recall": cm[i, i] / cm[i].sum(),
            "most_confused_with": STAGES[j] if errors[j] > 0 else "-",
            "confused_count": int(errors[j]),
        })
    return pd.DataFrame(rows)


def make_prediction_table(split, prob):
    meta = DATA[split]["meta"]
    pred = prob.argmax(axis=1)
    table = meta[["subject", "epoch_start", "epoch_end"]].copy()
    table["actual"] = [STAGES[i] for i in meta["label"]]
    table["predicted"] = [STAGES[i] for i in pred]
    table["confidence"] = prob.max(axis=1)
    for i, stage in enumerate(STAGES):
        table["prob_" + stage] = prob[:, i]
    table["correct"] = table["actual"] == table["predicted"]
    return table

## 7. ประเมินผล Train และ Validation (ไม่ประเมิน Test ใน Exp นี้)

In [ ]:
results = {}
tables = {}

for split in ["train", "val"]:
    X, y = DATA[split]["X"], DATA[split]["y"]
    prob = model.predict(X, batch_size=BATCH_SIZE, verbose=0)
    y_pred = prob.argmax(axis=1)

    overall, per_class = calc_metrics(y, y_pred)
    results[split] = overall
    tables[split] = make_prediction_table(split, prob)

    if split == "train":
        continue

    cm = confusion_matrix(y, y_pred, labels=LABELS)
    cm_norm = confusion_matrix(y, y_pred, labels=LABELS, normalize="true")

    print("=====", split.upper(), "=====")
    display(pd.DataFrame([overall]).round(4))

    print("Per-class metrics")
    display(per_class.round(4))

    print("Per-stage confusion (one-vs-rest)")
    display(stage_confusion(cm).round(4))

    print("Per-class analysis")
    display(class_analysis(cm).round(4))

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    ConfusionMatrixDisplay(cm, display_labels=STAGES).plot(ax=axes[0], cmap="Blues", colorbar=False)
    axes[0].set_title("Confusion matrix (count)")
    ConfusionMatrixDisplay(cm_norm, display_labels=STAGES).plot(ax=axes[1], cmap="Blues", values_format=".2f", colorbar=False)
    axes[1].set_title("Confusion matrix (normalized by actual)")
    plt.suptitle(split)
    plt.tight_layout()
    plt.savefig(RESULT_DIR / f"{split}_confusion_matrix.png", dpi=150)
    plt.show()

    x_pos = np.arange(len(STAGES))
    plt.figure(figsize=(8, 4.5))
    plt.bar(x_pos - 0.27, per_class["precision"], 0.27, label="Precision")
    plt.bar(x_pos, per_class["recall"], 0.27, label="Recall")
    plt.bar(x_pos + 0.27, per_class["f1_score"], 0.27, label="F1-score")
    plt.xticks(x_pos, STAGES)
    plt.ylim(0, 1.05)
    plt.title(f"{split}: metrics per stage")
    plt.legend()
    plt.grid(axis="y", alpha=0.4)
    plt.tight_layout()
    plt.savefig(RESULT_DIR / f"{split}_per_class_metrics.png", dpi=150)
    plt.show()

    pd.DataFrame([overall]).to_csv(RESULT_DIR / f"{split}_overall_metrics.csv", index=False)
    per_class.to_csv(RESULT_DIR / f"{split}_per_class_metrics.csv", index=False)
    stage_confusion(cm).to_csv(RESULT_DIR / f"{split}_per_stage_confusion.csv", index=False)
    class_analysis(cm).to_csv(RESULT_DIR / f"{split}_class_analysis.csv", index=False)
    pd.DataFrame(cm, index=STAGES, columns=STAGES).to_csv(RESULT_DIR / f"{split}_confusion_matrix.csv")
    tables[split].to_csv(RESULT_DIR / f"{split}_predictions.csv", index=False)

summary = pd.DataFrame(results).T
summary.index.name = "split"
summary.to_csv(RESULT_DIR / "baseline_summary.csv")
display(summary.round(4))

## 8. แถวสรุปผลสำหรับตารางบันทึกการทดลอง (Validation)

In [ ]:
summary_row = pd.DataFrame([{
    "experiment": "exp2b_cnn_7ch",
    "changed_factor": "Exp 2 + ReduceLROnPlateau + max epoch 60",
    "trainable_params": int(sum(np.prod(w.shape) for w in model.trainable_weights)),
    "best_val_loss": round(best_val_loss, 4),
    "val_accuracy": round(results["val"]["accuracy"], 4),
    "val_macro_f1": round(results["val"]["macro_f1"], 4),
    "val_kappa": round(results["val"]["cohen_kappa"], 4),
    "train_acc_minus_val_acc": round(results["train"]["accuracy"] - results["val"]["accuracy"], 4),
}])
display(summary_row)
summary_row.to_csv(RESULT_DIR / "experiment_row.csv", index=False)
